# Synthetic Probe Generation — Method 2: Neural Face Swap (inswapper_128)

## Why not the original SimSwap?

The original [neuralchen/SimSwap](https://github.com/neuralchen/SimSwap) repo pins
`insightface==0.2.1` and requires Python 3.6–3.9, conflicting with this project's
Python 3.10 + modern InsightFace environment. Running it requires a separate conda
environment and subprocess calls — significant overhead for ~150 probes.

**Used instead:** InsightFace's own `inswapper_128.onnx` — a GAN-based identity-injection
model from the same InsightFace ecosystem. It produces qualitatively different results
from Method 1's geometric warp: identity is re-rendered by the network rather than
geometrically transplanted, giving different artifact signatures relevant to the A/B case
stratification.

## Model download (one-time)

```bash
# ~555 MB
mkdir -p ~/.insightface/models
wget -O ~/.insightface/models/inswapper_128.onnx \
  https://huggingface.co/deepinsight/inswapper/resolve/main/inswapper_128.onnx
```

## Pipeline
- **Source**: random probe from a *different* identity (donor — provides facial identity)
- **Target**: gallery image of the *true* identity (recipient — provides geometry/background)
- **Result**: donor's face identity re-rendered by neural network onto target's face

**Output**: `data/synthetic_probes/simswap/{identity}/{identity}_simswap_{i}.jpg`  
**Metadata**: appended to `data/synthetic_probes/metadata.csv`

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

SPLIT_PATH      = "splits/lfw_1N_split.json"
GALLERY_EMB     = "cache/G.npy"
GALLERY_IDS     = "cache/gallery_ids.npy"
CRISE_SUMMARY   = "results/rise_alignedchip_baseline_multi/summary_K5_N1000_s8_p0.5_MASTERSEED123.csv"

OUT_DIR         = "data/synthetic_probes/simswap"
META_PATH       = "data/synthetic_probes/metadata.csv"   # shared with Method 1; we append
GEN_METHOD      = "simswap"                              # label written to metadata

SYNTH_SEED      = 42     # same seed → same 50 identities + same source pairings
N_TARGET_IDS    = 50
N_PROBES_PER_ID = 3

# Path to inswapper_128.onnx (override if stored elsewhere)
INSWAPPER_PATH  = None   # None → uses ~/.insightface/models/inswapper_128.onnx

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os, json
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from insightface.app import FaceAnalysis

from synth_gen import (
    load_inswapper,
    face_swap_neural,
    get_embedding_from_image,
    select_target_identities,
    select_sources_for_target,
)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(META_PATH), exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup + inswapper model
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("FaceAnalysis ready")

swapper = load_inswapper(INSWAPPER_PATH)
print(f"inswapper_128 loaded: {swapper}")

In [ ]:
# ---------------------------------------------------------------------------
# Load split + gallery
# ---------------------------------------------------------------------------

with open(SPLIT_PATH) as f:
    split = json.load(f)

G           = np.load(GALLERY_EMB).astype(np.float32)
gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()
id_to_index = {gid: i for i, gid in enumerate(gallery_ids)}

print(f"Gallery: {G.shape[0]} identities")

In [ ]:
# ---------------------------------------------------------------------------
# Select 50 target identities  (same seed → same set as Method 1)
# ---------------------------------------------------------------------------

rise_summary = pd.read_csv(CRISE_SUMMARY)
id_col = "true_id" if "true_id" in rise_summary.columns else rise_summary.columns[0]
completed_ids = rise_summary[id_col].dropna().unique().tolist()

target_ids = select_target_identities(completed_ids, n=N_TARGET_IDS, seed=SYNTH_SEED)
print(f"Target identities: {len(target_ids)} (seed={SYNTH_SEED})")
print("First 5:", target_ids[:5])

In [ ]:
# ---------------------------------------------------------------------------
# Main generation loop
# ---------------------------------------------------------------------------

records = []
n_ok = 0
n_fail = 0

for exp_i, target_id in enumerate(tqdm(target_ids, desc="Target identities")):
    target_gallery_path = split["gallery"][target_id]
    target_img = cv2.imread(target_gallery_path)

    if target_img is None:
        print(f"[WARN] Could not read gallery image for {target_id}")
        continue

    id_out_dir = os.path.join(OUT_DIR, target_id)
    os.makedirs(id_out_dir, exist_ok=True)

    source_pairs = select_sources_for_target(
        target_id, exp_i, split, n=N_PROBES_PER_ID, seed=SYNTH_SEED
    )

    for k, (src_id, src_path) in enumerate(source_pairs):
        source_img = cv2.imread(src_path)
        if source_img is None:
            n_fail += 1
            records.append(dict(
                identity=target_id, generation_method=GEN_METHOD,
                source_identity=src_id, source_path=src_path, blend_alpha=None,
                output_path=None, embedding_ok=False,
                arcface_similarity=None, rank1_match=None,
                saliency_cosine_sim=None, saliency_l1=None, case_label=None,
            ))
            continue

        swapped = face_swap_neural(source_img, target_img, app, swapper)

        if swapped is None:
            n_fail += 1
            records.append(dict(
                identity=target_id, generation_method=GEN_METHOD,
                source_identity=src_id, source_path=src_path, blend_alpha=None,
                output_path=None, embedding_ok=False,
                arcface_similarity=None, rank1_match=None,
                saliency_cosine_sim=None, saliency_l1=None, case_label=None,
            ))
            continue

        out_fname = f"{target_id}_simswap_{k}.jpg"
        out_path  = os.path.join(id_out_dir, out_fname)
        cv2.imwrite(out_path, swapped, [cv2.IMWRITE_JPEG_QUALITY, 95])

        n_ok += 1
        records.append(dict(
            identity=target_id, generation_method=GEN_METHOD,
            source_identity=src_id, source_path=src_path, blend_alpha=None,
            output_path=out_path, embedding_ok=True,
            arcface_similarity=None, rank1_match=None,
            saliency_cosine_sim=None, saliency_l1=None, case_label=None,
        ))

print(f"\nGeneration complete: {n_ok} saved, {n_fail} failed")

In [ ]:
# ---------------------------------------------------------------------------
# Visualisation: 6 examples — Source | Neural swap | Target (gallery)
# ---------------------------------------------------------------------------

ok_records = [r for r in records if r["output_path"] is not None]
show_targets = target_ids[:6]

fig, axes = plt.subplots(len(show_targets), 3,
                         figsize=(9, 3 * len(show_targets)))

for row, tid in enumerate(show_targets):
    rec_row = next((r for r in ok_records if r["identity"] == tid), None)
    if rec_row is None:
        for col in range(3):
            axes[row, col].axis("off")
        continue

    src_img  = cv2.cvtColor(cv2.imread(rec_row["source_path"]), cv2.COLOR_BGR2RGB)
    tgt_img  = cv2.cvtColor(cv2.imread(split["gallery"][tid]),  cv2.COLOR_BGR2RGB)
    swap_img = cv2.cvtColor(cv2.imread(rec_row["output_path"]), cv2.COLOR_BGR2RGB)

    for col, (img, title) in enumerate([
        (src_img,  f"Source\n{rec_row['source_identity'][:22]}"),
        (swap_img, "Neural swap (inswapper_128)"),
        (tgt_img,  f"Target (gallery)\n{tid[:22]}"),
    ]):
        axes[row, col].imshow(img)
        axes[row, col].set_title(title, fontsize=8)
        axes[row, col].axis("off")

plt.suptitle("Neural Swap (inswapper_128) — Source | Result | Target", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_grid.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Compute ArcFace embeddings
# ---------------------------------------------------------------------------

synth_embeddings = {}

for r in tqdm(ok_records, desc="Embedding synthetic probes"):
    emb = get_embedding_from_image(r["output_path"], app, rec)
    synth_embeddings[r["output_path"]] = emb
    if emb is None:
        r["embedding_ok"] = False

n_emb_ok   = sum(1 for e in synth_embeddings.values() if e is not None)
n_emb_fail = sum(1 for e in synth_embeddings.values() if e is None)
print(f"Embeddings: {n_emb_ok} OK, {n_emb_fail} failed")

In [ ]:
# ---------------------------------------------------------------------------
# Compute rank-1 match + cosine similarity to true identity
# ---------------------------------------------------------------------------

for r in records:
    if not r["embedding_ok"] or r["output_path"] is None:
        continue
    emb = synth_embeddings.get(r["output_path"])
    if emb is None:
        r["embedding_ok"] = False
        continue

    sims     = G @ emb
    true_idx = id_to_index[r["identity"]]
    r["arcface_similarity"] = round(float(sims[true_idx]), 6)
    r["rank1_match"]        = (int(np.argmax(sims)) == true_idx)

eval_rows  = [r for r in records if r["rank1_match"] is not None]
rank1_rate = sum(r["rank1_match"] for r in eval_rows) / max(len(eval_rows), 1)
mean_sim   = np.mean([r["arcface_similarity"] for r in eval_rows]) if eval_rows else float("nan")

print(f"Evaluable probes : {len(eval_rows)}")
print(f"Rank-1 match rate: {rank1_rate:.4f}  (fraction fooling ArcFace)")
print(f"Mean sim to true : {mean_sim:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Append to shared metadata.csv (created by Method 1 notebook)
# If file doesn't exist yet, create it; otherwise append without duplicates.
# ---------------------------------------------------------------------------

META_COLS = [
    "identity", "generation_method", "source_identity", "blend_alpha",
    "output_path", "embedding_ok", "arcface_similarity", "rank1_match",
    "saliency_cosine_sim", "saliency_l1", "case_label",
]

new_df = pd.DataFrame(records)[META_COLS]

if os.path.exists(META_PATH):
    existing = pd.read_csv(META_PATH)
    # Drop any stale simswap rows in case this cell is re-run
    existing = existing[existing["generation_method"] != GEN_METHOD]
    combined = pd.concat([existing, new_df], ignore_index=True)
else:
    combined = new_df

combined.to_csv(META_PATH, index=False)
print(f"metadata.csv: {len(combined)} total rows → {META_PATH}")
print(combined.groupby("generation_method")[["embedding_ok", "rank1_match"]].mean().round(3))

In [ ]:
# ---------------------------------------------------------------------------
# Quick stats for this method
# ---------------------------------------------------------------------------

if eval_rows:
    sims_all = [r["arcface_similarity"] for r in eval_rows]
    print("=== Neural swap (inswapper_128) stats ===")
    print(f"  Rank-1 rate : {rank1_rate:.4f}")
    print(f"  Sim mean    : {np.mean(sims_all):.4f}  std={np.std(sims_all):.4f}")
    print(f"  Sim > 0.3   : {sum(s > 0.3 for s in sims_all)} / {len(sims_all)}")
    print(f"  Sim > 0.4   : {sum(s > 0.4 for s in sims_all)} / {len(sims_all)}")